In [98]:
import os
import re
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

def ttest_performance(
    root_dir="./results/",
    target_topk=None,
    target_metrics=None,
    target_campus=None,
    target_loss_compare=None,
    highlight_pval=0.05
):
    """
    对 performance.txt 结果做两样本 t 检验并打印。
    baseline 始终在后，MeanDiff = lossX - baseline

    参数：
        root_dir: str, 根目录
        target_topk: list[str], 指定 TopK，为空时使用所有 TopK
        target_metrics: list[str], 指定指标，为空时使用所有 Metric
        target_campus: list[str], 指定校区，为空时使用所有 Campus
        target_loss_compare: list[tuple], 指定要比较的 loss 对，例如 [("loss3","loss0")]
        highlight_pval: float, 显著性标红阈值
    """

    # 正则表达式
    pattern_loss = re.compile(r"loss(\d+)")
    pattern_metric = re.compile(r"(.+?):([\d.]+)")

    # 解析 performance.txt
    def parse_performance_file(filepath):
        metrics = {}
        current_topk = None
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith("Top"):
                    current_topk = line
                    metrics[current_topk] = {}
                elif ":" in line and current_topk is not None:
                    match = pattern_metric.match(line)
                    if match:
                        key, val = match.groups()
                        metrics[current_topk][key.strip()] = float(val)
        return metrics

    # 读取所有文件
    records = []
    for model in os.listdir(root_dir):
        model_path = os.path.join(root_dir, model)
        if not os.path.isdir(model_path):
            continue

        for campus in os.listdir(model_path):
            campus_path = os.path.join(model_path, campus)
            if not os.path.isdir(campus_path):
                continue

            for fname in os.listdir(campus_path):
                if fname.endswith("-performance.txt"):
                    match = pattern_loss.search(fname)
                    if not match:
                        continue
                    loss_id = int(match.group(1))
                    filepath = os.path.join(campus_path, fname)
                    metrics = parse_performance_file(filepath)

                    for topk, vals in metrics.items():
                        for metric, v in vals.items():
                            records.append({
                                "Model": model,
                                "Campus": campus,
                                "Loss": f"loss{loss_id}",
                                "TopK": topk,
                                "Metric": metric,
                                "Value": v
                            })

    df_raw = pd.DataFrame(records)

    # 如果参数为空，则使用 DataFrame 中所有唯一值
    if target_topk is None or len(target_topk) == 0:
        target_topk = df_raw["TopK"].unique().tolist()
    if target_metrics is None or len(target_metrics) == 0:
        target_metrics = df_raw["Metric"].unique().tolist()
    if target_campus is None or len(target_campus) == 0:
        target_campus = df_raw["Campus"].unique().tolist()
    if target_loss_compare is None or len(target_loss_compare) == 0:
        target_loss_compare = [("loss3", "loss0"), ("loss5", "loss0")]

    # 打印表头
    header = f"{'Model':<10} {'Campus':<12} {'TopK':<7} {'Metric':<12} {'Compare':<15} {'MeanLossA':>10} {'MeanLossB':>10} {'MeanDiff':>10} {'t_stat':>8} {'p_val':>8} {'sig':>4}"
    print(header)
    print("-" * len(header))

    # 两样本 t 检验 & 打印
    for (campus, model, topk, metric), group in df_raw.groupby(["Campus", "Model", "TopK", "Metric"]):
        if topk not in target_topk or metric not in target_metrics or campus not in target_campus:
            continue

        # 收集每个 loss 的值
        values_by_loss = {loss: group[group["Loss"] == loss]["Value"].values for loss in group["Loss"].unique()}

        for lossX, baseline in target_loss_compare:
            # 确保 baseline 始终在后
            if lossX in values_by_loss and baseline in values_by_loss:
                valsX, valsB = values_by_loss[lossX], values_by_loss[baseline]
                meanX, meanB = valsX.mean(), valsB.mean()
                mean_diff = meanX - meanB
                t_stat, p_val = ttest_ind(valsX, valsB, equal_var=False)

                # 显著性星号
                if p_val < 0.01:
                    sig = "***"
                elif p_val < 0.05:
                    sig = "**"
                elif p_val < 0.1:
                    sig = "*"
                else:
                    sig = ""

                # 标红/加粗逻辑
                if meanX > meanB:
                    if p_val < highlight_pval:
                        # 红色 + 加粗
                        # line_prefix = "\033[91;4m"
                        line_prefix = "\033[91m"
                    else:
                        # 红色
                        line_prefix = "\033[91m"
                else:
                    line_prefix = ""
                line_suffix = "\033[0m" if line_prefix else ""

                # 打印
                line = f"{model:<10} {campus:<12} {topk:<7} {metric:<12} {lossX+' vs '+baseline:<15} {meanX:>10.4f} {meanB:>10.4f} {mean_diff:>10.4f} {t_stat:>8.3f} {p_val:>8.4f} {sig:>4}"
                print(f"{line_prefix}{line}{line_suffix}")


In [99]:
import os
import re
import pandas as pd
import numpy as np
from collections import defaultdict

# ================== 工具函数 ==================
# 颜色渲染：正数红色，其余默认
def colorize(val: float, width=10):
    if pd.isna(val):
        return " " * width
    s = f"{val:.3f}".rjust(width)  # 固定宽度，保证对齐
    if val > 0:
        return f"\033[91m{s}\033[0m"  # 红色
    return s

# 表格打印（对齐 + 颜色）
def print_colored_table(df: pd.DataFrame, title: str):
    print(title)
    # 打印列名
    header = " " * 10 + "".join(c.rjust(10) for c in df.columns)
    print(header)
    # 打印每行
    for idx, row in df.iterrows():
        row_str = str(idx).ljust(10)
        for val in row:
            row_str += colorize(val, width=10)
        print(row_str)
    print()


# ================== 主函数 ==================
def print_improvement_table(
    root_dir="./results/",
    baseline_loss="loss0",
    value_type="AbsDiff",       # "AbsDiff" 或 "RelDiff(%)"
    target_campus=None,
    target_loss=None,
    target_topk=None,
    target_metrics=None
):
    pattern_loss = re.compile(r"loss(\d+)")
    pattern_metric = re.compile(r"(.+?):([\d.]+)")

    # 解析 performance.txt
    def parse_performance_file(filepath):
        metrics = {}
        current_topk = None
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith("Top"):
                    current_topk = line
                    metrics[current_topk] = {}
                elif ":" in line and current_topk is not None:
                    match = pattern_metric.match(line)
                    if match:
                        key, val = match.groups()
                        metrics[current_topk][key.strip()] = float(val)
        return metrics

    # ------------------
    # 读取所有文件
    # ------------------
    results = []
    for model in os.listdir(root_dir):
        model_path = os.path.join(root_dir, model)
        if not os.path.isdir(model_path):
            continue
        for campus in os.listdir(model_path):
            if target_campus is not None and campus not in target_campus:
                continue
            campus_path = os.path.join(model_path, campus)
            if not os.path.isdir(campus_path):
                continue

            # 收集每个 loss 的 performance.txt
            raw_loss_metrics = defaultdict(list)
            for fname in os.listdir(campus_path):
                if fname.endswith("-performance.txt"):
                    match = pattern_loss.search(fname)
                    if not match:
                        continue
                    loss_id = int(match.group(1))
                    loss_name = f"loss{loss_id}"
                    if target_loss is not None and loss_name not in target_loss + [baseline_loss]:
                        continue
                    filepath = os.path.join(campus_path, fname)
                    raw_loss_metrics[loss_id].append(parse_performance_file(filepath))

            # 合并同 loss 多文件，取均值
            loss_metrics = {}
            for loss_id, metrics_list in raw_loss_metrics.items():
                merged = {}
                for metrics in metrics_list:
                    for topk, vals in metrics.items():
                        if topk not in merged:
                            merged[topk] = defaultdict(list)
                        for metric, v in vals.items():
                            merged[topk][metric].append(v)
                # 每个 metric 取均值
                loss_metrics[loss_id] = {
                    topk: {m: np.mean(vs) for m, vs in mdict.items()}
                    for topk, mdict in merged.items()
                }

            # 计算提升值
            for topk in loss_metrics.get(1, {}).keys() if target_topk is None else target_topk:
                baseline = loss_metrics.get(0, {}).get(topk, {})
                if not baseline:
                    continue
                for loss_id, metrics_dict in loss_metrics.items():
                    if loss_id == 0:
                        continue
                    if target_loss is not None and f"loss{loss_id}" not in target_loss:
                        continue
                    compare = metrics_dict.get(topk, {})
                    for metric, value in compare.items():
                        if target_metrics is not None and metric not in target_metrics:
                            continue
                        base_val = baseline.get(metric, None)
                        if base_val is not None and base_val != 0:
                            abs_diff = value - base_val
                            rel_diff = abs_diff / base_val * 100
                            results.append({
                                "Model": model,
                                "Campus": campus,
                                "Loss": f"loss{loss_id}",
                                "TopK": topk,
                                "Metric": metric,
                                "Baseline": base_val,
                                "Value": value,
                                "AbsDiff": abs_diff,
                                "RelDiff(%)": rel_diff
                            })

    df = pd.DataFrame(results)

    if df.empty:
        print("No data found!")
        return

    # ------------------
    # 打印表格
    # ------------------
    df[value_type] = pd.to_numeric(df[value_type], errors="coerce")
    loss_keep = target_loss if target_loss is not None else sorted(df["Loss"].unique())
    topk_list = target_topk if target_topk is not None else sorted(df["TopK"].unique())

    for campus, df_c in df.groupby("Campus", sort=True):
        # print(f"\n########## Campus: {campus} ##########")
        for loss in loss_keep:
            df_l = df_c[df_c["Loss"] == loss]
            for topk in topk_list:
                g = df_l[df_l["TopK"] == topk]
                if g.empty:
                    continue
                pivot = (
                    g.groupby(["Metric", "Model"])[value_type]
                     .mean()
                     .reset_index()
                     .pivot(index="Metric", columns="Model", values=value_type)
                     .sort_index()
                     .round(3)
                )
                pivot = pivot.reindex(sorted(pivot.columns), axis=1)  # 模型列排序
                # 打印表格
                print_colored_table(
                    pivot,
                    f"=== Results | Campus={campus} | Loss={loss} | TopK={topk} ({value_type}) ==="
                )

# 打印相较于Baseline的均值差/提升百分比

In [100]:
print_improvement_table(
    root_dir="./repeat_results/results2",
    baseline_loss="loss0",
    value_type="RelDiff(%)", # "AbsDiff" 或 "RelDiff(%)"
    # target_campus=["campus_15"],
    target_loss=["loss3"],
    target_topk=["Top 10"],
    # target_metrics=["Hit Ratio","NDCG"]
)

=== Results | Campus=campus_10 | Loss=loss3 | TopK=Top 10 (RelDiff(%)) ===
                 NCL       SGL    SimGCL   XSimGCL
Hit Ratio     -0.251    -0.040    -0.012    -0.375
NDCG           0.377    -0.296    -0.464    -0.191
Precision     -0.249    -0.036    -0.008    -0.375
Recall        -0.321    -0.477    -0.877    -0.607

=== Results | Campus=campus_102 | Loss=loss3 | TopK=Top 10 (RelDiff(%)) ===
                 NCL       SGL    SimGCL   XSimGCL
Hit Ratio     -0.266     0.528     0.088    -0.355
NDCG           0.294     0.593     0.635    -0.018
Precision     -0.266     0.531     0.091    -0.349
Recall         0.102     1.023     0.510     0.014

=== Results | Campus=campus_143 | Loss=loss3 | TopK=Top 10 (RelDiff(%)) ===
                 NCL       SGL    SimGCL   XSimGCL
Hit Ratio      0.182     0.833     1.316     1.159
NDCG          -0.867     0.032    -0.715     0.606
Precision      0.180     0.834     1.323     1.155
Recall         0.113     1.142     1.425     1.428

=== R

# t检验

In [143]:
ttest_performance(
    # root_dir="./results/",
    root_dir="./repeat_results/results1",
    target_topk=["Top 10"],
    # target_metrics=["Hit Ratio", "NDCG"],
    target_campus=["campus_15"], # 有10、15、34、102、143
    target_loss_compare=[("loss3","loss0")],
    highlight_pval=0.05
)


Model      Campus       TopK    Metric       Compare          MeanLossA  MeanLossB   MeanDiff   t_stat    p_val  sig
--------------------------------------------------------------------------------------------------------------------
NCL        campus_15    Top 10  Hit Ratio    loss3 vs loss0      0.3467     0.3466     0.0001    0.242   0.8143     
NCL        campus_15    Top 10  NDCG         loss3 vs loss0      0.3698     0.3700    -0.0002   -1.087   0.3043     
NCL        campus_15    Top 10  Precision    loss3 vs loss0      0.1680     0.1679     0.0000    0.244   0.8128     
NCL        campus_15    Top 10  Recall       loss3 vs loss0      0.4346     0.4341     0.0005    0.883   0.4050     
SGL        campus_15    Top 10  Hit Ratio    loss3 vs loss0      0.3474     0.3471     0.0004    0.480   0.6438     
SGL        campus_15    Top 10  NDCG         loss3 vs loss0      0.3625     0.3645    -0.0020   -1.603   0.1608     
SGL        campus_15    Top 10  Precision    loss3 vs loss0     